# Demo 2 — A2A

Two A2A agents in a chain: you → **equity** → **macro**. Each agent has its own LLM and its own MCP tools.

```
   you (notebook on localhost)
        │
        │  A2A
        ▼
   equity  (127.0.0.3:9999) ──A2A──► macro  (127.0.0.2:9998)
        │                   ◄─reply──
        ▼
   combined brief back to you
```

The notebook hosts no agents — both agents live on different loopback IPs to make their separateness visually obvious. Equity is independent of macro; it just *also* acts as an A2A client when it needs to delegate.

## First-time setup

Run this once in a terminal, from the directory where you want the repo:

```bash
git clone https://github.com/jackwu502/ivado-protocol.git
cd ivado-protocol

python3.12 -m venv .venv
source .venv/bin/activate

python -m pip install --upgrade pip
python -m pip install -r requirements.txt
python -m ipykernel install --user --name ivado-lab --display-name "ivado-lab (3.12)"
```

Create your local `.env` file:

```bash
cp .env.example .env
```

Then edit `.env` and fill in one credential route:

```bash
# Option 1: Anthropic direct
ANTHROPIC_API_KEY=sk-ant-...
ANTHROPIC_MODEL=claude-sonnet-4-6

# Option 2: OpenRouter-compatible Anthropic endpoint
# ANTHROPIC_BASE_URL=https://openrouter.ai/api
# ANTHROPIC_API_KEY=sk-or-v1-...
# ANTHROPIC_MODEL=anthropic/claude-sonnet-4.5
```

Do not commit `.env`; it is intentionally gitignored.

Start Jupyter from the repo root and select the `ivado-lab (3.12)` kernel:

```bash
python -m jupyter lab
```


Use the same `.env` from Demo 1.

In [1]:
# ── Bootstrap: resolve paths to shared/ and sibling helpers ──
import sys
from pathlib import Path
HERE = Path.cwd()
ROOT = HERE.parent
sys.path.insert(0, str(ROOT))   # so `from shared.X import Y` works
sys.path.insert(0, str(HERE))   # so sibling helpers import directly
STOCK_MCP_SERVER = str(ROOT / "shared" / "stock_mcp_server.py")


In [2]:
%pip install -q "a2a-sdk<1.0" uvicorn httpx anthropic mcp yfinance python-dotenv

from dotenv import load_dotenv
load_dotenv(ROOT / ".env")
print("ready")

Note: you may need to restart the kernel to use updated packages.
ready


## 1. Imports

The user only knows one address — equity's. Macro's URL is hidden inside equity's own source.

In [3]:
import asyncio
import json
import uuid
import httpx
import uvicorn
from a2a.server.agent_execution import AgentExecutor, RequestContext
from a2a.server.apps import A2AStarletteApplication
from a2a.server.events import EventQueue
from a2a.server.request_handlers import DefaultRequestHandler
from a2a.server.tasks import InMemoryTaskStore
from a2a.types import AgentCapabilities, AgentCard, AgentSkill
from a2a.utils import new_agent_text_message
from shared.agent_runner import run_agent

# The only address the *user* (this notebook) ever needs.
# Notebook runs on localhost; both agents live on different loopback IPs
# (equity on 127.0.0.3, macro on 127.0.0.2) so they look like remote services.
EQUITY_URL = "http://127.0.0.3:9999"
print("ready — user only knows:", EQUITY_URL)

ready — user only knows: http://127.0.0.3:9999


## 2. Macro analyst

The leaf agent. Its own Claude + MCP tools, wrapped as an A2A server so equity can call it.

In [4]:
# Macro picks its own bind address + port. Using 127.0.0.2 (a
# distinct loopback IP) so visually it looks like a separate service.
MACRO_HOST = "127.0.0.2"
MACRO_PORT = 9998

MACRO_SYSTEM_PROMPT = """You are a macro / sector analyst. Given a sector
or thematic question, answer concisely (2-3 sentences) using the available
stock-data tools if helpful. Do not produce long reports."""


class MacroAnalystExecutor(AgentExecutor):
    async def execute(self, context: RequestContext, event_queue: EventQueue) -> None:
        question = context.get_user_input() or "Give a brief market outlook."
        macro_trace: list[str] = []
        try:
            answer = await run_agent(
                question=question,
                mcp_servers=[STOCK_MCP_SERVER],
                system_prompt=MACRO_SYSTEM_PROMPT,
                trace=macro_trace,
                agent_label="macro",
            )
        except Exception as exc:
            answer = f"(macro analyst error: {exc})"
        # Bundle macro's trace lines into the reply so equity can interleave
        # them into its own trace at the right point in time.
        if macro_trace:
            payload = (
                "\u27ea\u27eaTRACE\u27eb\u27eb\n"
                + "\n".join(macro_trace)
                + "\n\u27ea\u27ea/TRACE\u27eb\u27eb\n"
                + answer
            )
        else:
            payload = answer
        await event_queue.enqueue_event(new_agent_text_message(payload))

    async def cancel(self, context, event_queue):
        raise NotImplementedError


def build_macro_card() -> AgentCard:
    return AgentCard(
        name="MacroAnalystAgent",
        description="Sector / macro outlook analyst.",
        url=f"http://{MACRO_HOST}:{MACRO_PORT}/",
        version="1.0.0",
        default_input_modes=["text"],
        default_output_modes=["text"],
        capabilities=AgentCapabilities(streaming=False),
        skills=[AgentSkill(
            id="sector_outlook",
            name="Sector outlook",
            description="Quick view of a sector or macro theme.",
            tags=["finance", "macro", "sector"],
            examples=["semiconductor sector outlook", "energy in 2025"],
        )],
    )

## 3. Equity analyst (chain caller)

Same as macro, plus one extra tool: `ask_macro_analyst`. When Claude calls it, equity makes an A2A request to macro — that's the chain edge.

In [5]:
# Equity's own bind address + port. Using 127.0.0.3 (a distinct
# loopback IP from macro's 127.0.0.2) so neither agent shares an address
# with the notebook, and the trace makes it visible they're separate services.
EQUITY_HOST = "127.0.0.3"
EQUITY_PORT = 9999

EQUITY_SYSTEM_PROMPT = """You are an equity analyst writing a brief on a
single stock. Use the stock-data tools to fetch price action, company info,
and news. If sector or macro context would inform your brief, call
`ask_macro_analyst` ONCE for it. Then write a concise brief."""


ASK_MACRO_TOOL = {
    "name": "ask_macro_analyst",
    "description": (
        "Ask the macro/sector analyst for sector or thematic context. "
        "Use at most once per brief."
    ),
    "input_schema": {
        "type": "object",
        "properties": {
            "question": {
                "type": "string",
                "description": "Plain-English question for the macro analyst.",
            }
        },
        "required": ["question"],
    },
}


import re


def _extract_macro_trace(reply_text: str):
    """If reply was wrapped with ⟪⟪TRACE⟫⟫...⟪⟪/TRACE⟫⟫, pull the trace
    lines out and return (trace_lines, cleaned_text)."""
    m = re.search(
        r"\u27ea\u27eaTRACE\u27eb\u27eb\n(.*?)\n\u27ea\u27ea/TRACE\u27eb\u27eb\n",
        reply_text, flags=re.S,
    )
    if not m:
        return [], reply_text
    return m.group(1).splitlines(), reply_text[:m.start()] + reply_text[m.end():]


def _make_call_macro_a2a(equity_trace: list[str]):
    """Build an A2A-client closure that appends macro's trace lines into
    equity's trace at the moment the macro call returns — so the audience
    sees [equity] -> [macro] tool calls -> [equity] in temporal order."""
    MACRO_URL = "http://127.0.0.2:9998"

    async def call_macro_a2a(name_, args):
        if name_ != "ask_macro_analyst":
            return f"(unknown tool: {name_})"
        question = args.get("question", "")
        print(f"  [equity → A2A] calling MacroAnalystAgent at {MACRO_URL}: \"{question[:60]}\"")
        request = {
            "jsonrpc": "2.0",
            "id": str(uuid.uuid4()),
            "method": "message/send",
            "params": {
                "message": {
                    "role": "user",
                    "messageId": str(uuid.uuid4()),
                    "parts": [{"kind": "text", "text": question}],
                }
            },
        }
        async with httpx.AsyncClient(timeout=120.0) as client:
            r = await client.post(f"{MACRO_URL}/", json=request)
            resp = r.json()
        result = resp.get("result", {}) or {}
        if result.get("kind") == "message":
            for part in result.get("parts", []):
                if part.get("kind") == "text":
                    raw = part["text"]
                    macro_lines, clean = _extract_macro_trace(raw)
                    # Splice macro's trace into equity's trace right here,
                    # between equity's [tool] line and equity's [result] line.
                    if macro_lines:
                        equity_trace.extend(macro_lines)
                    print(f"  [equity ← A2A] reply ({len(clean)} chars, "
                          f"+{len(macro_lines)} trace lines from macro)")
                    return clean
        return "(no text reply from macro)"

    return call_macro_a2a


class EquityAnalystExecutor(AgentExecutor):
    async def execute(self, context: RequestContext, event_queue: EventQueue) -> None:
        question = context.get_user_input() or "Analyze the market."
        trace: list[str] = []
        call_macro_a2a = _make_call_macro_a2a(trace)
        try:
            answer = await run_agent(
                question=question,
                mcp_servers=[STOCK_MCP_SERVER],            # MCP layer
                system_prompt=EQUITY_SYSTEM_PROMPT,
                extra_tools=[ASK_MACRO_TOOL],              # A2A chain edge
                extra_tool_executor=call_macro_a2a,
                trace=trace,
                agent_label="equity",
            )
        except Exception as exc:
            answer = f"(equity analyst error: {exc})"
        if trace:
            full = (
                "── Internal trace (each line shows which agent called the tool) ──\n"
                + "\n".join(trace)
                + "\n── End trace ──\n\n"
                + answer
            )
        else:
            full = answer
        await event_queue.enqueue_event(new_agent_text_message(full))

    async def cancel(self, context, event_queue):
        raise NotImplementedError


def build_equity_card() -> AgentCard:
    return AgentCard(
        name="EquityAnalystAgent",
        description="Equity analyst — produces stock briefs; may chain to a macro analyst.",
        url=f"http://{EQUITY_HOST}:{EQUITY_PORT}/",
        version="1.0.0",
        default_input_modes=["text"],
        default_output_modes=["text"],
        capabilities=AgentCapabilities(streaming=False),
        skills=[AgentSkill(
            id="stock_brief",
            name="Stock brief",
            description="Brief on a single stock; may include sector context.",
            tags=["finance", "equities", "research"],
            examples=["Analyze NVDA briefly. Include sector context."],
        )],
    )

## 4. Boot both agents

Two `uvicorn` tasks. **Notebook is on localhost; both agents live on different loopback IPs** — equity on `127.0.0.3`, macro on `127.0.0.2` — so the trace makes it obvious the two agents are remote services from the user's perspective.

> **macOS first-run only:** if you see *"Can't assign requested address"*, alias both loopbacks once (sticks until reboot):
> ```bash
> sudo ifconfig lo0 alias 127.0.0.2 up
> sudo ifconfig lo0 alias 127.0.0.3 up
> ```

In [6]:
def _build_app(card, executor):
    handler = DefaultRequestHandler(
        agent_executor=executor, task_store=InMemoryTaskStore(),
    )
    return A2AStarletteApplication(agent_card=card, http_handler=handler).build()


async def _start(app, host, port):
    config = uvicorn.Config(app, host=host, port=port, log_level="warning")
    server = uvicorn.Server(config)
    server.install_signal_handlers = lambda: None
    asyncio.create_task(server.serve())
    for _ in range(40):
        await asyncio.sleep(0.1)
        if server.started: break
    return server

macro_server  = await _start(_build_app(build_macro_card(),  MacroAnalystExecutor()),
                             MACRO_HOST, MACRO_PORT)
equity_server = await _start(_build_app(build_equity_card(), EquityAnalystExecutor()),
                             EQUITY_HOST, EQUITY_PORT)
print(f"MacroAnalystAgent  on http://{MACRO_HOST}:{MACRO_PORT}")
print(f"EquityAnalystAgent on http://{EQUITY_HOST}:{EQUITY_PORT}")

MacroAnalystAgent  on http://127.0.0.2:9998
EquityAnalystAgent on http://127.0.0.3:9999


## 5. The user role

Quick sanity-check: from the user's perspective, the only address in scope is `EQUITY_URL`. Macro exists, but it's hidden behind equity — exactly the encapsulation A2A is designed for. The user never has to learn that there's even a chain happening.

In [7]:
from a2a_helpers import send_message, show_response

print("Where each thing lives:")
print(f"  localhost          ← notebook (the user). NO agent here.")
print(f"  127.0.0.3:9999     ← EquityAnalystAgent. Notebook calls this.")
print(f"  127.0.0.2:9998     ← MacroAnalystAgent. ONLY equity calls this; notebook never touches it.")

Where each thing lives:
  localhost          ← notebook (the user). NO agent here.
  127.0.0.3:9999     ← EquityAnalystAgent. Notebook calls this.
  127.0.0.2:9998     ← MacroAnalystAgent. ONLY equity calls this; notebook never touches it.


## 6. Send one sentence — watch the chain

We send **one** sentence to equity and wait for one final brief. While the call is in flight, watch the printed trace: you'll see equity's own tool calls, then `[equity → A2A]` when its Claude decides to delegate, then macro's tool calls interleaved live, then `[equity ← A2A]` when macro replies, then equity weaves it all into the brief.

In [8]:
response = await send_message(
    "Analyze NVDA briefly. Include sector context.",
    url=EQUITY_URL,
    timeout=180.0,
)
show_response(response)

AGENT:
  ── Internal trace (each line shows which agent called the tool) ──
  [equity] [tool]   get_quote(ticker=NVDA)
  [equity] [result] {   "ticker": "NVDA",   "price": 198.4499969482422,   "change": -1.1200103759765625,   "change_pct": -0.561211772747499…
  [equity] [tool]   get_company_info(ticker=NVDA)
  [equity] [result] {   "ticker": "NVDA",   "name": "NVIDIA Corporation",   "sector": "Technology",   "industry": "Semiconductors",   "mark…
  [equity] [tool]   get_history(ticker=NVDA, days=30)
  [equity] [result] {   "ticker": "NVDA",   "dates": [     "2026-03-20",     "2026-03-23",     "2026-03-24",     "2026-03-25",     "2026-03…
  [equity] [tool]   get_news_headlines(ticker=NVDA, limit=5)
  [equity] [result] {   "ticker": "NVDA",   "headlines": [     {       "title": "AI Chipmakers in Korea, Taiwan Drive Asian Stocks to Recor…
  [equity] [tool]   ask_macro_analyst(question=What's the current outlook for the semiconductor sector, particularly AI chip m…)
  [macro] [tool]   get_

### Recap

Two architectural points worth pausing on:

1. **Equity is both server (to you) and client (to macro).** That's the whole A2A chain pattern in one sentence — every agent is a peer that can also call other agents.
2. **Look at the duplicate `get_quote(NVDA)` lines** — equity called it, then macro called it again. Why? Because each A2A call is a fresh Claude conversation; macro's context is empty when it starts and has no idea equity already fetched that data. Compare with Demo 1's MCP, where everything sits in one `messages[]` array and Claude never re-asks. **This is the price of A2A's encapsulation: callee = separate brain.**

And the limitation: `MACRO_URL` is still hardcoded in equity's source. A2A spec doesn't standardize a registry, so every chain edge gets wired by hand. Demo 3 replaces that hardcode with a runtime lookup.

## 7. Raw JSON-RPC envelope (optional)

In [9]:
show_response(response, raw=True)

{
  "id": "dbd9d60a-f1f6-4b35-8220-ca48ef746c04",
  "jsonrpc": "2.0",
  "result": {
    "kind": "message",
    "messageId": "70a272d5-f484-49a0-b27d-0a864e608608",
    "parts": [
      {
        "kind": "text",
        "text": "── Internal trace (each line shows which agent called the tool) ──\n[equity] [tool]   get_quote(ticker=NVDA)\n[equity] [result] {   \"ticker\": \"NVDA\",   \"price\": 198.4499969482422,   \"change\": -1.1200103759765625,   \"change_pct\": -0.561211772747499…\n[equity] [tool]   get_company_info(ticker=NVDA)\n[equity] [result] {   \"ticker\": \"NVDA\",   \"name\": \"NVIDIA Corporation\",   \"sector\": \"Technology\",   \"industry\": \"Semiconductors\",   \"mark…\n[equity] [tool]   get_history(ticker=NVDA, days=30)\n[equity] [result] {   \"ticker\": \"NVDA\",   \"dates\": [     \"2026-03-20\",     \"2026-03-23\",     \"2026-03-24\",     \"2026-03-25\",     \"2026-03…\n[equity] [tool]   get_news_headlines(ticker=NVDA, limit=5)\n[equity] [result] {   \"ticker\": \"NV

## 8. Stop both agents

In [10]:
for srv in (equity_server, macro_server):
    srv.should_exit = True
await asyncio.sleep(1)
print("both agents stopped")

both agents stopped
